## Concept 8 — Plan-and-Execute Agent

### What is it?
For complex multi-step queries, a single agent can lose track of the goal.
Plan-and-Execute separates work into three roles:
1. **Planner** — LLM chain that generates a step-by-step plan
2. **Executor** — a `create_agent` that executes each step autonomously
3. **Synthesizer** — LLM chain that combines all step results into a final answer

### vs ReAct (Notebook 2)
```
ReAct (reactive):         One step at a time, doesn't know what's next.
                          Good for 1-2 tool calls.

Plan-and-Execute:         Full plan generated BEFORE any execution.
                          Each step knows what came before.
                          Good for ordered multi-step tasks.
```

### Pipeline
```
Query
  → Planner LLM  → [Step 1, Step 2, Step 3]
  → Executor (create_agent):
       Step 1 → result_1
       Step 2 + result_1 → result_2
       Step 3 + result_1 + result_2 → result_3
  → Synthesizer LLM → Final coherent answer
```

### Limitation (what Concept 9 fixes)
No quality check on the final answer.
→ Concept 9 (Self-Correcting) validates the answer and retries if it fails.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = get_llm(temperature=0.0)
print(f'LLM ready | Tools: {[t.name for t in TOOLS]}')

### Step 1 — Planner (LLM Chain)
Breaks the query into an ordered numbered list of concrete steps.

In [ ]:
planner_chain = ChatPromptTemplate.from_messages([
    ('system',
     'You are a planning assistant. Break the query into ordered steps.\n'
     'Available tools: calculate (math), search_docs (LangChain docs), get_weather (weather).\n'
     'Output ONLY a numbered list. Each step is ONE concrete action.\n'
     'Example output:\n'
     '1. Call search_docs to find what RAG means\n'
     '2. Call calculate to compute 25 * 4'),
    ('human', 'Query: {query}\n\nPlan:'),
]) | llm | StrOutputParser()

test_q = 'Search docs for RAG and also calculate 25 multiplied by 4'
plan   = planner_chain.invoke({'query': test_q})
print(f'Query: {test_q}\n\nPlan generated:\n{plan}')

### Step 2 — Executor (create_agent)
A proper agent executes each step. It receives the step text + context from prior steps.

In [ ]:
executor_agent = create_agent(
    model=llm,
    tools=TOOLS,
    system_prompt=(
        'You execute one specific step using the available tools. '
        'You will be given the step to execute and results from previous steps. '
        'Use tools as instructed. Return only the result of this step.'
    ),
)

def execute_step(step: str, prior_context: str) -> str:
    prompt = f'Execute this step: {step}'
    if prior_context:
        prompt += f'\n\nResults from previous steps:\n{prior_context}'

    result   = executor_agent.invoke({'messages': [{'role': 'user', 'content': prompt}]})
    answer   = result['messages'][-1].content

    # Show what tools the executor called
    for msg in result['messages']:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for call in msg.tool_calls:
                print(f'    [Executor called] {call["name"]}({call["args"]})')
    return answer

# Test with step 1
steps = [l.strip() for l in plan.strip().split('\n') if l.strip() and l.strip()[0].isdigit()]
print(f'Parsed steps: {steps}\n')
print(f'Step 1 result: {execute_step(steps[0], "")}')

### Step 3 — Synthesizer (LLM Chain)
Combines all step results into one coherent final answer.

In [ ]:
synthesizer_chain = ChatPromptTemplate.from_messages([
    ('system', 'Synthesize the step results into a clear, complete final answer.'),
    ('human', 'Original query: {query}\n\nStep results:\n{step_results}\n\nFinal answer:'),
]) | llm | StrOutputParser()

### Step 4 — Full Plan-and-Execute Loop
Planner → Executor (create_agent) × N steps → Synthesizer → Answer

In [ ]:
def plan_and_execute(query: str) -> str:
    print(f'\n[Planner] Generating plan...')
    plan_text = planner_chain.invoke({'query': query})

    steps = [l.strip() for l in plan_text.strip().split('\n')
             if l.strip() and (l.strip()[0].isdigit() or l.strip().startswith('-'))]
    if not steps:
        steps = [plan_text]

    print(f'[Planner] {len(steps)} step(s): {steps}')

    step_results = []
    context      = ''

    for i, step in enumerate(steps, 1):
        print(f'\n[Executor] Step {i}: {step}')
        result = execute_step(step, context)
        step_results.append(f'Step {i}: {result}')
        context = '\n'.join(step_results)
        short = result[:100] + '...' if len(result) > 100 else result
        print(f'  Result: {short}')

    print(f'\n[Synthesizer] Combining {len(step_results)} result(s)...')
    return synthesizer_chain.invoke({'query': query, 'step_results': context})

print('=== Plan-and-Execute Demo ===')
answer = plan_and_execute('Search docs for RAG and also calculate 25 multiplied by 4')
print(f'\n=== FINAL ANSWER ===\n{answer}')

### Full Function

In [ ]:
def plan_execute_agent(query: str) -> str:
    return plan_and_execute(query)

### Test — Standard Queries

In [ ]:
run_queries(plan_execute_agent, label='Concept 8 — Plan-and-Execute Agent')